# Neuro-Symbolic Reasoning with LLMs

In this notebook we look at **neuro-symbolic reasoning in the era of LLMs**.

We will:

- Build **intuition** for what *neuro-symbolic* means.
- See **where LLMs struggle** and how symbolic methods help.
- Implement **small, realistic examples** where:
  - an LLM handles **unstructured language**, and
  - a symbolic layer (rules / graphs / code) enforces **logic, constraints, and guarantees**.

This can be your **first unit on neuro-symbolic ideas** – not just abstract philosophy, but **patterns you can actually use** in real-world applications.


## 1. What Is Neuro-Symbolic AI? (LLM Era View)

**Neuro-symbolic AI** combines:

- **Neural components** – continuous, statistical, fuzzy (LLMs, embeddings, deep nets)  
- **Symbolic components** – discrete, structured, rule-based (logic, programs, graphs, constraints)

You can think of it as:

```text
Neural side   = pattern recognition, language understanding, intuition
Symbolic side = rules, logic, constraints, planning, guarantees
```

Recent work shows that **LLMs already have some latent symbolic abilities**, but often in a *fragile* way, and they benefit from **explicit symbolic control**:

- *Large Language Models Are Neurosymbolic Reasoners* – Fang et al., AAAI 2024
- *LINC: A Neurosymbolic Approach for Logical Reasoning by Combining Language Models with First-Order Logic Provers* – Olausson et al., 2023
- Surveys on **neuro-symbolic AI** and **data- & knowledge-driven AI** – Yu et al., Nawaz et al., and others
- Work on **symbolic control** over LLMs (e.g., Ctrl-G, logical constraints at generation time)
- **Program-of-Thought / code prompting** as a neural-symbolic pattern: LLM writes code, symbolic executor runs it

### Why this matters in practice

Pure LLM systems can:

- hallucinate facts
- violate safety / compliance rules
- be inconsistent across turns
- fail on multi-step constraints (dates, budgets, eligibility, etc.)

**Neuro-symbolic systems** treat the LLM as a *front-end* to a more structured reasoner:

```text
User Text
   ↓
LLM: parse / draft / propose
   ↓
Symbolic Layer: check / enforce / compute / prove
   ↓
Final Answer with guarantees (or clear failure)
```

In this notebook we’ll build **three mini-patterns** that show how to do this in code.


### Imports (for all examples)

We only use standard Python + minimal helpers so this stays lightweight and easy to adapt.


In [ ]:
from __future__ import annotations  # Enables forward-referencing for type hints within class definitions.

from dataclasses import dataclass     # Decorator that automatically generates boilerplate methods like __init__ and __repr__.
from typing import List, Dict, Tuple, Optional, Callable, Any  # Standard tools for defining specific data types for static analysis.
import textwrap  # Utility for formatting, wrapping, and dedenting multi-line text strings.

## 2. Tiny Primer: Symbols, Facts, and Rules

In symbolic AI we often talk about **facts** and **rules**:

- **Fact**: a small atomic statement that is either true or false under some interpretation.  
  - Example: `age(alice, 16)`, `role(alice, "intern")`, `country(alice, "USA")`  
- **Rule**: a conditional statement over facts.  
  - Example:  
    > if age < 18 then cannot_approve_transactions_over(1000)

In code, we don’t need a full logic engine to get value.  
We can represent facts as **Python data structures** and rules as **functions** that:

- read these structures
- return booleans or structured conclusions

This is already *neuro-symbolic* when coupled with an LLM that **extracts facts from messy text**.


## 3. Example 1 – LLM + Rule Engine for Policy Compliance

### Real-world use case

Imagine an internal tool that checks whether a proposed **access request** or **change request** obeys company policy.  
The input is natural language, but the policy is written as clear rules:

> *"Interns cannot deploy to production.  
> Junior engineers may deploy only with approval.  
> Senior engineers can deploy up to risk level 3 without extra approvals."*

**Neuro-symbolic pattern**:

```text
Natural language request
   ↓
LLM: extract structured facts (role, risk_level, approvals, etc.)
   ↓
Python rules: check policy, compute decision + explanation
   ↓
Safe, consistent decision
```

Below we’ll *simulate* the LLM extraction step with hard-coded structured data (so the notebook runs anywhere), but you can imagine the extraction coming from an LLM call.


In [ ]:
# ----- Symbolic layer: data structures -----

@dataclass
class DeploymentRequest:
    actor: str          # person making the request
    role: str           # e.g., "intern", "junior", "senior"
    risk_level: int     # 1 (low) .. 5 (very high)
    has_approval: bool  # e.g., from tech lead or manager


@dataclass
class PolicyDecision:
    allowed: bool
    reasons: List[str]


def evaluate_policy(req: DeploymentRequest) -> PolicyDecision:
    """Apply simple symbolic rules to a deployment request.

    This function does *not* care about how we got the request fields;
    it only enforces the policies.
    """
    reasons: List[str] = []

    # Rule 1: interns cannot deploy to production at all
    if req.role.lower() == "intern":
        reasons.append("Interns are not allowed to deploy to production.")
        return PolicyDecision(allowed=False, reasons=reasons)

    # Rule 2: junior engineers need approval for any deployment
    if req.role.lower() == "junior" and not req.has_approval:
        reasons.append("Junior engineers require explicit approval to deploy.")
        return PolicyDecision(allowed=False, reasons=reasons)

    # Rule 3: senior engineers can deploy up to risk level 3 without approval
    if req.role.lower() == "senior" and req.risk_level <= 3:
        reasons.append("Senior engineer deploying at acceptable risk level (<=3)." )
        return PolicyDecision(allowed=True, reasons=reasons)

    # Rule 4: any deployment with risk > 3 requires approval, regardless of role
    if req.risk_level > 3 and not req.has_approval:
        reasons.append("High-risk deployments (>3) require additional approval.")
        return PolicyDecision(allowed=False, reasons=reasons)

    # Default: allow if none of the above blocked it
    reasons.append("No policy rule blocked this deployment.")
    return PolicyDecision(allowed=True, reasons=reasons)

### Simulating the LLM extraction

In a real system, you’d prompt an LLM with something like:

> *"Extract actor, role, risk_level (1-5), and whether explicit approval is present from the following ticket description. Return JSON."*

Here we will just simulate the result as if the LLM already did this.


In [ ]:
def pretty_print_decision(req: DeploymentRequest, dec: PolicyDecision) -> None:
    print("Request:")
    print(f"  actor       : {req.actor}")
    print(f"  role        : {req.role}")
    print(f"  risk_level  : {req.risk_level}")
    print(f"  has_approval: {req.has_approval}")

    print("\nDecision:")
    print("  allowed     :", dec.allowed)
    print("  reasons     :")
    for r in dec.reasons:
        print("   -", r)
    print("\n" + "-" * 60 + "\n")


# Example natural language tickets (in reality, LLM would parse these)
ticket_1 = """
As an intern on the payments team, I want to hotfix a minor issue directly in production.
"""

ticket_2 = """
I'm a junior engineer fixing a critical bug (risk level 5). 
I already have approval from my tech lead to deploy.
"""

ticket_3 = """
Senior backend engineer deploying a routine config change (risk level 2),
no explicit approval in the ticket.
"""

# Simulated LLM outputs (structured extractions)
req1 = DeploymentRequest(actor="Alice", role="intern", risk_level=2, has_approval=False)
req2 = DeploymentRequest(actor="Bob", role="junior", risk_level=5, has_approval=True)
req3 = DeploymentRequest(actor="Carol", role="senior", risk_level=2, has_approval=False)

for req in [req1, req2, req3]:
    decision = evaluate_policy(req)
    pretty_print_decision(req, decision)

**Key point**: the **symbolic layer** is:

- small,
- explicit,
- auditable,
- testable.

The LLM’s job is *only* to interpret messy human language into structured fields.  
This makes the overall system:

- more robust,
- easier to debug,
- easier to explain to auditors / regulators.


## 4. Example 2 – LLM + Tiny Knowledge Graph

### Real-world use case

Suppose you have a **company knowledge graph** with entities and relations:

- people, teams, products
- who works where
- who owns which system
- who is on-call for what

An LLM can:

- read messy questions, like:  
  *"Who owns the payments API?"*  
  *"Who is the backup owner for checkout?"*
- convert them into **structured queries** over the graph  
- and the **symbolic graph layer** executes those queries exactly.

Here we will:

1. Build a tiny in-memory knowledge graph.  
2. Show how a *parsed* query can be answered symbolically.  
3. Simulate what an LLM would output as that parsed query.


In [ ]:
# ----- Symbolic layer: a tiny knowledge graph -----

@dataclass
class Edge:
    source: str
    relation: str
    target: str


class TinyKG:
    def __init__(self, edges: List[Edge]) -> None:
        self.edges = edges

    def neighbors(self, node: str, relation: Optional[str] = None) -> List[str]:
        """Return targets reachable from `node` via given relation (or any relation if None)."""
        results = []
        for e in self.edges:
            if e.source == node and (relation is None or e.relation == relation):
                results.append(e.target)
        return results

    def find(self, relation: str, target: str) -> List[str]:
        """Find all sources such that source -[relation]-> target."""
        return [e.source for e in self.edges if e.relation == relation and e.target == target]

    def debug_print(self) -> None:
        print("Knowledge Graph edges:")
        for e in self.edges:
            print(f"  {e.source} -[{e.relation}]-> {e.target}")


# Build a tiny company KG
edges = [
    Edge("payments-api", "owned_by", "team-payments"),
    Edge("checkout-service", "owned_by", "team-checkout"),
    Edge("team-payments", "manager", "Dana"),
    Edge("team-checkout", "manager", "Eli"),
    Edge("team-payments", "backup_owner", "Frank"),
]

kg = TinyKG(edges)
kg.debug_print()

### Simulated LLM query parsing

Again, in a real application you’d prompt the LLM with something like:

> *"You are a query parser. Given a user question about system ownership, output JSON with `intent`, `entity`, and optional `role`."*

We’ll simulate that by hand and focus on the symbolic part: **exact lookup in the knowledge graph**.


In [ ]:
def answer_question(parsed_query: Dict[str, str]) -> str:
    """Symbolic answering given a parsed query.

    Expected keys in parsed_query:
      - intent: e.g., "get_owner", "get_manager", "get_backup_owner"
      - entity: system/team name, e.g., "payments-api" or "team-payments"
    """
    intent = parsed_query.get("intent")
    entity = parsed_query.get("entity")

    if intent == "get_owner":
        owners = kg.neighbors(entity, relation="owned_by")
        if not owners:
            return f"No owner found for {entity}."
        return f"{entity} is owned by: {', '.join(owners)}"

    if intent == "get_manager":
        managers = kg.neighbors(entity, relation="manager")
        if not managers:
            return f"No manager recorded for {entity}."
        return f"Manager for {entity}: {', '.join(managers)}"

    if intent == "get_backup_owner":
        backups = kg.neighbors(entity, relation="backup_owner")
        if not backups:
            return f"No backup owner recorded for {entity}."
        return f"Backup owner for {entity}: {', '.join(backups)}"

    return "I don't know how to handle that intent symbolically."


# Simulated LLM parses for three questions
questions_and_parses = [
    (
        "Who owns the payments API?",  # user natural language
        {"intent": "get_owner", "entity": "payments-api"},
    ),
    (
        "Who manages the payments team?", 
        {"intent": "get_manager", "entity": "team-payments"},
    ),
    (
        "Who is the backup owner for the payments team?", 
        {"intent": "get_backup_owner", "entity": "team-payments"},
    ),
]

for question, parsed in questions_and_parses:
    print("Question:", question)
    print("Parsed query (from LLM):", parsed)
    print("Symbolic KG answer:", answer_question(parsed))
    print("-" * 60)

## 5. Example 3 – Program-of-Thought (LLM writes code, Python reasons)

Another powerful pattern is **Program-of-Thought (PoT)** or **code prompting**:

- Ask the LLM to **write a program** that solves the problem.
- Execute the program in a **sandboxed interpreter**.
- Use the program’s output as the final answer.

This is a neural-symbolic method: the neural model proposes *symbolic code*, and a deterministic executor runs it.

We won’t actually call an LLM here (to keep this notebook self-contained), but we’ll show:

1. A natural language task.  
2. An example of code an LLM *could* generate.  
3. How running that code gives a precise result.


In [ ]:
problem = """
You have a budget of $5,000 to run an event.

You must rent a venue for $2,000.
Each attendee costs $35 (food, swag, etc.).
You also want a 15% safety buffer on top of all costs.

Question: What is the maximum number of attendees you can support?
Ignore taxes and other fees.
"""

print(textwrap.fill(problem, width=80))

### Example: “LLM-generated” code

A good LLM, prompted with *"Write a Python function that computes the answer"*, might produce something like:


In [ ]:
# This cell simulates code that an LLM could generate.
# In a real system, you'd treat this as an untrusted string and execute it
# in a restricted / sandboxed environment.

def max_attendees(budget: float = 5000.0,
                  venue_cost: float = 2000.0,
                  per_attendee_cost: float = 35.0,
                  safety_buffer_fraction: float = 0.15) -> int:
    """Compute the maximum number of attendees given the constraints.

    Explanation:
      - Total raw cost = venue_cost + per_attendee_cost * n
      - With safety buffer, total_cost = raw_cost * (1 + safety_buffer_fraction)
      - We need total_cost <= budget, solve for n.
    """
    # Solve inequality: (venue + 35*n) * 1.15 <= 5000
    # => 35*n <= (5000 / 1.15) - 2000
    allowed_raw_cost = budget / (1 + safety_buffer_fraction)
    remaining_for_attendees = allowed_raw_cost - venue_cost
    n = remaining_for_attendees // per_attendee_cost  # floor to integer
    return max(0, int(n))  # just in case

print("Maximum attendees:", max_attendees())

## 6. Visual Summary – Neuro-Symbolic Patterns with LLMs

### 6.1 Policy Compliance (Extraction + Rules)

```text
Ticket text
   ↓
LLM: extract (role, risk_level, approvals, ...)
   ↓
Symbolic rules in Python
   ↓
Allow / Block + Explanation
```

### 6.2 Knowledge Graph Q&A

```text
User question
   ↓
LLM: parse to (intent, entity, ...)
   ↓
Knowledge graph / ontology
   ↓
Exact lookup, joins, constraints
```

### 6.3 Program-of-Thought / Code Prompting

```text
Problem in natural language
   ↓
LLM: generate code / program
   ↓
Symbolic executor (Python, solver, prover)
   ↓
Checked, reproducible result
```

Across all three patterns, the **LLM is the front-end** that:

- understands language,
- proposes structure,

and the **symbolic back-end**:

- enforces logic,
- connects to real data,
- can be tested, verified, and explained.


## 7. When Neuro-Symbolic Helps

**Helps a lot when:**

- You must obey **hard constraints** (safety, compliance, SLAs).  
- There is a **source-of-truth database / KG / ruleset** separate from the LLM.  
- You need **auditability** and **consistent decisions** over time.  
- Tasks mix **messy language** with **clean structure** (forms, tickets, contracts, logs).

Neuro-Symbolic AI is a very active area of research right now and central to building **reliable, controllable AI systems** on top of LLMs.
